# IHP SG13G2 Palace–HFSS inductor benchmark

**Objective.** Reproduce the public three-turn, two-port SG13G2 spiral with finite-conductivity Palace boundaries and compare the full complex S matrix to the supplied HFSS 3D Layout export.

**Hypothesis.** With the `EMDesign2` stack, vertical Metal1-to-TopMetal1 gap ports, and a converged mesh, Palace should reproduce the 1.5–3.5 GHz HFSS response without fitting geometry or conductivity.

The default execution downloads and verifies the small source files, checks the reference, and builds the model. Set `GSIM_EIC_RUN_MESH=1` for local meshing and `GSIM_EIC_RUN_CLOUD=1` to submit one two-excitation Palace job. Cloud execution is deliberately opt-in.

In [ ]:
# Copyright 2026 GDSFactory

from __future__ import annotations

import importlib.metadata
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from gsim.palace.benchmarks import (
    IHP_GDS_ARTIFACT,
    IHP_HFSS_REFERENCE_ARTIFACT,
    differential_inductance_quality,
    download_artifact,
    from_palace_sparams,
    load_touchstone_2port,
    maximum_singular_value,
    power_loss_fraction,
    reciprocity_error,
    sparameter_error_summary,
)
from gsim.palace.benchmarks.eic_ihp import (
    IHP_EXPECTED_BBOX_UM,
    IHP_PORT_SPECS,
    IHP_SOURCE_COMMIT,
    build_ihp_stack,
    load_ihp_component,
    make_ihp_simulation,
)


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Run this notebook from within the gsim repository")


repo_root = find_repo_root()
cache_dir = repo_root / ".cache" / "eic" / "ihp"
run_cloud = os.getenv("GSIM_EIC_RUN_CLOUD") == "1"
run_mesh = os.getenv("GSIM_EIC_RUN_MESH") == "1" or run_cloud
print(
    {"gsim": importlib.metadata.version("gsim"), "mesh": run_mesh, "cloud": run_cloud}
)

## Pinned public inputs and acceptance status

The GDS and Touchstone files come from commit [`a98509e`](https://github.com/SkillSurf/frac-n-pll-vco-smacd_2026/tree/a98509eb57989f60e7965bee88214d081d53c27e/HFSS%20Inductor%20Files). The source repository declares no license, so these artifacts are downloaded at runtime and never redistributed here.

The Touchstone header identifies HFSS 3D Layout and the frozen geometry variables, but omits the design name. It is therefore treated as the published reference, with the handoff's final acceptance gate still open: an ANSYS user must reopen `Feb_01st.aedt`, select `EMDesign2`, apply the frozen variables, and confirm a fresh export is identical.

In [ ]:
gds_path = download_artifact(IHP_GDS_ARTIFACT, cache_dir / IHP_GDS_ARTIFACT.name)
touchstone_path = download_artifact(
    IHP_HFSS_REFERENCE_ARTIFACT,
    cache_dir / IHP_HFSS_REFERENCE_ARTIFACT.name,
)
hfss = load_touchstone_2port(touchstone_path)

assert len(hfss.frequency_hz) == 41
assert np.allclose(np.diff(hfss.frequency_hz), 0.05e9)
center_index = int(np.flatnonzero(np.isclose(hfss.frequency_hz, 2.45e9))[0])
inductance_h, quality_factor = differential_inductance_quality(hfss)
np.testing.assert_allclose(inductance_h[center_index] * 1e9, 4.005845, rtol=1e-6)
np.testing.assert_allclose(quality_factor[center_index], 16.2104, rtol=1e-5)

print("source commit:", IHP_SOURCE_COMMIT)
print("S(2.45 GHz):\n", hfss.s[center_index])
print(
    f"Ldiff={inductance_h[center_index] * 1e9:.6f} nH, Qdiff={quality_factor[center_index]:.4f}"
)
print(f"reciprocity error={reciprocity_error(hfss):.3g}")
print(f"largest singular value={maximum_singular_value(hfss):.6f}")
print(
    "unscattered power range:",
    power_loss_fraction(hfss).min(),
    power_loss_fraction(hfss).max(),
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
frequency_ghz = hfss.frequency_hz / 1e9
for label, row, column in (("S11", 0, 0), ("S12", 0, 1), ("S21", 1, 0), ("S22", 1, 1)):
    values = hfss.s[:, row, column]
    axes[0].plot(frequency_ghz, 20 * np.log10(np.abs(values)), label=label)
    axes[1].plot(frequency_ghz, np.unwrap(np.angle(values)) * 180 / np.pi, label=label)
axes[0].set(xlabel="Frequency (GHz)", ylabel="Magnitude (dB)")
axes[1].set(xlabel="Frequency (GHz)", ylabel="Unwrapped phase (degrees)")
for axis in axes:
    axis.grid(alpha=0.3)
    axis.legend()
fig.suptitle("Published HFSS reference")
fig.tight_layout()

## Palace model

The source layer numbers are not the foundry drawing-layer numbers: `(62, 0)` is the HFSS Metal1 reference plane, `(123, 0)` is the Metal5 underpass, `(125, 0)` is TopVia1, and `(126, 0)` is TopMetal1. The ports below are the exact feed-end rectangles resolved from `EMDesign2`; both are excited so Palace emits all four S entries in one job.

In [ ]:
component = load_ihp_component(gds_path)
stack = build_ihp_stack()
np.testing.assert_allclose(
    component.bbox_np(),
    np.asarray(IHP_EXPECTED_BBOX_UM).reshape(2, 2),
)
assert stack.validate_stack().valid

print("bbox (um):\n", component.bbox_np())
print("ports:", IHP_PORT_SPECS)
print("finite conductivities (S/m):")
for layer_name in ("Metal1", "Metal5", "TopVia1", "TopMetal1"):
    material = stack.layers[layer_name].material
    print(f"  {layer_name}: {stack.materials[material]['conductivity']:.6g}")

In [ ]:
simulation = make_ihp_simulation(
    gds_path,
    repo_root / "palace-sim-eic-ihp",
    adaptive_tol=5e-3,
    adaptive_max_samples=8,
)
simulation.numerical.order = 1
assert simulation.validate_config().valid
palace_result = None
mesh_result = None
print("Configured 41 points, 1.5-3.5 GHz, two 50-ohm vertical excitations.")

## Optional mesh and run

The checked local mesh uses 3 µm conductor refinement, a 120 µm maximum size, and first-order elements. The adaptive sweep is capped at eight samples with a `5e-3` tolerance, based on the bounded smoke run. `planar_conductors=False` is essential: Metal1, Metal5, and TopMetal1 must appear under `Boundaries.Conductivity`, while TopVia1 remains a conductive volume. A run is only submitted after `validate_mesh()` passes.

In [ ]:
if run_mesh:
    mesh_result = simulation.mesh(
        preset="coarse",
        refined_mesh_size=3.0,
        max_mesh_size=120.0,
        planar_conductors=False,
        verbose=False,
    )
    config_path = simulation.write_config()
    validation = simulation.validate_mesh()
    config = json.loads(config_path.read_text())
    conductivity_boundaries = config["Boundaries"]["Conductivity"]
    assert len(conductivity_boundaries) == 6
    print(validation)
    print(mesh_result.mesh_stats)
    print("conductivity boundaries:", conductivity_boundaries)
else:
    print("Mesh skipped; set GSIM_EIC_RUN_MESH=1 to reproduce the local validation.")

In [ ]:
if run_cloud:
    palace_result = simulation.run(check_cache=True, verbose="status")
    print("cloud job id:", simulation._job_id)
else:
    print("Cloud submission skipped; set GSIM_EIC_RUN_CLOUD=1 explicitly.")

In [ ]:
if palace_result is not None:
    palace = from_palace_sparams(palace_result)
    np.testing.assert_allclose(palace.frequency_hz, hfss.frequency_hz, rtol=1e-12)
    parity = sparameter_error_summary(hfss, palace)
    print(json.dumps(parity, indent=2))
    print(f"Palace reciprocity error={reciprocity_error(palace):.3g}")
    print(f"Palace largest singular value={maximum_singular_value(palace):.6f}")
    print(
        "Palace unscattered power range:",
        power_loss_fraction(palace).min(),
        power_loss_fraction(palace).max(),
    )
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True)
    for data, style, prefix in ((hfss, "-", "HFSS"), (palace, "--", "Palace")):
        for label, row, column in (
            ("S11", 0, 0),
            ("S12", 0, 1),
            ("S21", 1, 0),
            ("S22", 1, 1),
        ):
            values = data.s[:, row, column]
            axes[0].plot(
                data.frequency_hz / 1e9,
                20 * np.log10(np.abs(values)),
                style,
                label=f"{prefix} {label}",
            )
            axes[1].plot(
                data.frequency_hz / 1e9,
                np.unwrap(np.angle(values)) * 180 / np.pi,
                style,
                label=f"{prefix} {label}",
            )
    axes[0].set(xlabel="Frequency (GHz)", ylabel="Magnitude (dB)")
    axes[1].set(xlabel="Frequency (GHz)", ylabel="Unwrapped phase (degrees)")
    for axis in axes:
        axis.grid(alpha=0.3)
        axis.legend(ncol=2)
    fig.tight_layout()
else:
    print("Parity table will be emitted after an opt-in Palace run completes.")

## Findings and decision log

- The pinned HFSS data pass the stored 2.45 GHz S, Ldiff, and Qdiff checks.
- The Palace geometry uses the native `EMDesign2` z extents and measured finite conductivities; no parameter has been tuned to the reference.
- Local topology validation confirms two vertical ports, six finite-conductor shell groups, and the conductive TopVia1 volume.
- Production job `01a05b88-3e9d-7a01-8394-926de5b9a4a7` (`cbfdfbac...`, Palace `4930e88`) completed the 41-point, order-one, eight-sample adaptive sweep in 12m20s. Maximum `|delta S|` is 0.0487 for reflection and 0.0318 for transmission; reciprocity error is `1.0e-13`, largest singular value is 0.9719, and unscattered power is 9.5% to 12.0%.
- Exact-frequency smoke job `01a05b6c-c245-7731-bee2-acdaec0a1ba3` used the 3 um mesh. Targeted 2 um refinement job `01a05b95-16e0-7780-8443-15fd279df0d0` (`abfe9abd...`) completed in 8m04s; its maximum change is 0.0136 for reflection and 0.0237 for transmission. The result is therefore not yet mesh-converged.
- The native-ANSYS re-export acceptance gate remains open: `Feb_01st.aedt` must be reopened at the frozen `EMDesign2` variables before the supplied Touchstone file is called authoritative.